In [1]:
import os
import sys
print(os.getcwd())
sys.path.append(os.path.abspath('..'))

D:\Things\used_car\notebooks


# PHASE 4: Hands-on normalization

## Post Phase 2 DF

In [4]:
import pandas as pd
import json

processed_df = pd.read_parquet("../data/raw/processed_df.parquet")

In [5]:
print(processed_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 389309 entries, 0 to 389308
Data columns (total 21 columns):
 #   Column        Non-Null Count   Dtype              
---  ------        --------------   -----              
 0   id            389309 non-null  int64              
 1   region        389309 non-null  str                
 2   price         389309 non-null  int64              
 3   year          389309 non-null  float64            
 4   manufacturer  374376 non-null  str                
 5   model         384708 non-null  str                
 6   condition     241645 non-null  str                
 7   cylinders     229443 non-null  str                
 8   fuel          387006 non-null  str                
 9   odometer      387082 non-null  float64            
 10  title_status  381879 non-null  str                
 11  transmission  387548 non-null  str                
 12  drive         269890 non-null  str                
 13  type          303843 non-null  str                
 14 

## Post Qwen3 Imputation

In [7]:
with open("../data/json/llm_impute_predictions.json", "r", encoding="utf-8") as f:
    predictions = json.load(f)

pred_df = pd.DataFrame(predictions)

In [8]:
TARGETS = [
    "manufacturer",
    "model",
    "condition",
    "cylinders",
    "drive",
    "type",
]

merged_df = processed_df.merge(
    pred_df[
        ["id"] + TARGETS
    ],
    on="id",
    how="left",
    suffixes=("", "_llm"),
)

for col in TARGETS:

    mask = (
        merged_df[col].isna()
        & merged_df[f"{col}_llm"].notna()
    )

    merged_df.loc[mask, col] = merged_df.loc[
        mask,
        f"{col}_llm"
    ]

for col in TARGETS:
  merged_df[col] = merged_df[col].apply(lambda x: str(x).lower().strip())

In [9]:
print(merged_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 389309 entries, 0 to 389308
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   id                389309 non-null  int64              
 1   region            389309 non-null  str                
 2   price             389309 non-null  int64              
 3   year              389309 non-null  float64            
 4   manufacturer      389309 non-null  str                
 5   model             389309 non-null  str                
 6   condition         389309 non-null  str                
 7   cylinders         389309 non-null  str                
 8   fuel              387006 non-null  str                
 9   odometer          387082 non-null  float64            
 10  title_status      381879 non-null  str                
 11  transmission      387548 non-null  str                
 12  drive             389309 non-null  str                


## Post GPT 5.6 Luna Normalization

In [11]:
with open('../data/json/manufacturer_mapping.json', 'r', encoding="utf-8") as file:
    manufacturer_mapping = json.load(file)


merged_df["manufacturer"] = (
    merged_df["manufacturer"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(manufacturer_mapping)
)
merged_df["manufacturer"] = (
    merged_df["manufacturer"]
    .replace(["nan", "none", "null"], pd.NA)
)

print(merged_df["manufacturer"].nunique())
print(merged_df["manufacturer"].value_counts(dropna=False))

43
manufacturer
ford               65084
chevrolet          51015
toyota             31686
honda              21209
jeep               17485
nissan             17368
ram                16415
gmc                15498
bmw                13320
dodge              12457
mercedes-benz      10296
other               9594
hyundai             9461
volkswagen          9061
subaru              9037
lexus               7553
kia                 7488
audi                7122
cadillac            6616
chrysler            5834
acura               5712
buick               5189
mazda               5136
infiniti            4489
lincoln             4016
volvo               3298
mitsubishi          3139
pontiac             2270
mini                2239
land rover          1997
jaguar              1915
porsche             1291
mercury             1152
saturn              1074
tesla                852
alfa-romeo           849
fiat                 777
harley-davidson      140
ferrari               84
datsun   

## Manual normalization

In [13]:
condition_mapping = {
    "good": "good",
    "excellent": "excellent",
    "fair": "fair",
    "like new": "like new",
    "new": "new",
    "salvage": "salvage",
    "very excellent": "excellent",

    "very good": "good",
    "great": "good",
    "plus good": "good",

    "mint": "like new",
    "mint condition": "like new",
    "perfect": "like new",
    "pristine": "like new",
    "certified": "like new",
    "certified pre-owned": "like new",
    "pre-owned": "like new",
    "pre owned": "like new",
    "one-owner": "like new",

    "used": "good",
    "use": "good",
    "driver": "fair",

    "as is": "fair",
    "as-is": "fair",

    "project": "salvage",
    "project as is": "salvage",
    "mid-project": "salvage",
    "restoration project": "salvage",
    "parts only": "salvage",
    "junk": "salvage",
    "nonrunner": "salvage",

    "sold": pd.NA,
    "unknown": pd.NA,
    "other": pd.NA,
    "any": pd.NA,
    "null": pd.NA,
    "nan": pd.NA,
    "us": pd.NA,
}

cylinder_mapping = {
    "3 cylinders": "3 cylinders",
    "4 cylinders": "4 cylinders",
    "5 cylinders": "5 cylinders",
    "6 cylinders": "6 cylinders",
    "8 cylinders": "8 cylinders",
    "10 cylinders": "10 cylinders",
    "12 cylinders": "12 cylinders",

    "7 cylinders": "other",
    "16 cylinders": "other",
    "2 cylinders": "other",
    "1 cylinder": "other",
    "0 cylinders": "other",
    "50 cc": "other",

    "other": "other",
    "null": pd.NA,
    "nan": pd.NA,
}

drive_mapping = {
    "fwd": "fwd",
    "rwd": "rwd",
    "4wd": "4wd",

    "awd": "4wd",
    "all wheel drive": "4wd",

    "2wd": pd.NA,
    "2 wd": pd.NA,
    "2 wheel drive": pd.NA,
    "4x2": pd.NA,
    "2x4": pd.NA,

    "auto": pd.NA,
    "automatic": pd.NA,

    "other": pd.NA,
    "null": pd.NA,
    "nan": pd.NA,
}

type_mapping = {
    "pickup": "pickup",
    "truck": "truck",
    "suv": "suv",
    "sedan": "sedan",
    "coupe": "coupe",
    "cupe": "coupe",
    "hatchback": "hatchback",
    "mini-van": "mini-van",
    "minivan": "mini-van",
    "convertible": "convertible",
    "wagon": "wagon",
    "van": "van",
    "offroad": "offroad",
    "bus": "bus",
    "other": "other",

    "roadster": "convertible",
    "crossover": "suv",
    "mpv": "mini-van",
    "car": "other",
    "limousine": "sedan",

    "stake body": "truck",
    "utility": "truck",
    "utility body": "truck",
    "sideloader": "truck",

    "travel trailer": "other",
    "trailer": "other",
    "hard shell pop up": "other",
    "motorcycle": "other",
    "tractor": "other",
    "cruiser": "other",

    "null": pd.NA,
    "nan": pd.NA,
}

merged_df["condition"] = (
    merged_df["condition"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(condition_mapping)
)

merged_df["cylinders"] = (
    merged_df["cylinders"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(cylinder_mapping)
)

merged_df["type"] = (
    merged_df["type"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(type_mapping)
)

merged_df["drive"] = (
    merged_df["drive"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(drive_mapping)
)

In [14]:
for col in ["condition", "cylinders", "drive", "type"]:
    print(col)
    print(sorted(processed_df[col].dropna().unique()))
    print(sorted(merged_df[col].dropna().unique()))
print(merged_df.info())

condition
['excellent', 'fair', 'good', 'like new', 'new', 'salvage']
['excellent', 'fair', 'good', 'like new', 'new', 'salvage']
cylinders
['10 cylinders', '12 cylinders', '3 cylinders', '4 cylinders', '5 cylinders', '6 cylinders', '8 cylinders', 'other']
['10 cylinders', '12 cylinders', '3 cylinders', '4 cylinders', '5 cylinders', '6 cylinders', '8 cylinders', 'other']
drive
['4wd', 'fwd', 'rwd']
['4wd', 'fwd', 'rwd']
type
['bus', 'convertible', 'coupe', 'hatchback', 'mini-van', 'offroad', 'other', 'pickup', 'sedan', 'suv', 'truck', 'van', 'wagon']
['bus', 'convertible', 'coupe', 'hatchback', 'mini-van', 'offroad', 'other', 'pickup', 'sedan', 'suv', 'truck', 'van', 'wagon']
<class 'pandas.DataFrame'>
RangeIndex: 389309 entries, 0 to 389308
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   id                389309 non-null  int64              
 1   region            389309 non-n

## Model normalization

In [16]:
import json

with open("../data/json/gpt_output_normalization_1.json") as f:
    model_mapping = json.load(f)

print(len(model_mapping))

clean_model_mapping = {}
manufacturer_inference = {}

for key, value in model_mapping.items():

    if isinstance(value, dict):
        clean_model_mapping[key] = value["model"]
        manufacturer_inference[key] = value["manufacturer"]

    else:
        clean_model_mapping[key] = value


merged_df["model"] = (
    merged_df["model"]
    .astype(str)
    .str.strip()
    .str.lower()
    .replace(clean_model_mapping)
)

for item in TARGETS:
  print(item)
  print(merged_df[item].nunique())

27245
manufacturer
43
model
2104
condition
6
cylinders
8
drive
3
type
13


## Filling paint color

In [18]:
merged_df["paint_color"] = (
    merged_df["paint_color"]
    .fillna("unknown")
    .str.strip()
    .str.lower()
)

merged_df["model"] = merged_df["model"].fillna("unknown")

# Lat lon 

# Dropping columns

In [21]:
print(merged_df.info())

<class 'pandas.DataFrame'>
RangeIndex: 389309 entries, 0 to 389308
Data columns (total 27 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   id                389309 non-null  int64              
 1   region            389309 non-null  str                
 2   price             389309 non-null  int64              
 3   year              389309 non-null  float64            
 4   manufacturer      389309 non-null  str                
 5   model             389309 non-null  object             
 6   condition         378295 non-null  str                
 7   cylinders         378359 non-null  str                
 8   fuel              387006 non-null  str                
 9   odometer          387082 non-null  float64            
 10  title_status      381879 non-null  str                
 11  transmission      387548 non-null  str                
 12  drive             376788 non-null  str                


## Feature engineering

In [23]:
merged_df["posting_date"] = pd.to_datetime(merged_df["posting_date"], utc=True)

In [24]:
from datetime import datetime
example = merged_df["posting_date"].iloc[0]
print(example.year)
print(example.month)
print(example.day)
print(example.hour)
print(example.weekday)
if example.weekday() < 5:
    print("It's a weekday!")
else:
    print("It's the weekend!")

2021
5
4
17
<bound method Timestamp.weekday of Timestamp('2021-05-04 17:31:18+0000', tz='UTC')>
It's a weekday!


In [25]:
merged_df["posting_year"] = merged_df["posting_date"].dt.year
merged_df["posting_month"] = merged_df["posting_date"].dt.month
merged_df["posting_day"] = merged_df["posting_date"].dt.day
merged_df["posting_hour"] = merged_df["posting_date"].dt.hour
merged_df["posting_weekday"] = merged_df["posting_date"].dt.weekday

In [26]:
print(merged_df[merged_df["year"] > merged_df["posting_year"]][["year","posting_year","manufacturer","model"]])

merged_df["vehicle_age"] = (
    merged_df["posting_year"] - merged_df["year"]
).clip(lower=0)

          year  posting_year manufacturer          model
9323    2022.0          2021        other        unknown
38656   2022.0          2021       toyota        4Runner
58818   2022.0          2021        honda          Civic
58819   2022.0          2021       toyota        unknown
61967   2022.0          2021   mitsubishi        Eclipse
...        ...           ...          ...            ...
374070  2022.0          2021   mitsubishi  Eclipse Cross
374071  2022.0          2021   mitsubishi  Eclipse Cross
375012  2022.0          2021   mitsubishi  Eclipse Cross
376590  2022.0          2021   mitsubishi  Eclipse Cross
385613  2022.0          2021        other        unknown

[103 rows x 4 columns]


In [27]:
merged_df["miles_per_year"] = (
    merged_df["odometer"] /
    (merged_df["vehicle_age"] + 1)
)

# Removing invalid lats and lons

In [29]:
original_len = len(merged_df)

merged_df = merged_df[
    merged_df["lat"].between(24.52, 49.38) &
    merged_df["long"].between(-124.77, -66.95)
].copy()

print(f"Removed {original_len - len(merged_df)} rows.")
print(f"Remaining: {len(merged_df)}")

Removed 6149 rows.
Remaining: 383160


# Dropping unnecessary rows

In [31]:
drop_columns = [
    "year",            # Replaced by vehicle_age
    "image_url",       # Not modelling images
    "description",     # Not using NLP
    "posting_date",    # Already decomposed
    "posting_year",    # Constant (2021)

    # Temporary LLM outputs
    "manufacturer_llm",
    "model_llm",
    "condition_llm",
    "cylinders_llm",
    "drive_llm",
    "type_llm",
]

print(merged_df.info())

final_df = merged_df.drop(columns=drop_columns)
print(final_df.info())

<class 'pandas.DataFrame'>
Index: 383160 entries, 0 to 389308
Data columns (total 34 columns):
 #   Column            Non-Null Count   Dtype              
---  ------            --------------   -----              
 0   id                383160 non-null  int64              
 1   region            383160 non-null  str                
 2   price             383160 non-null  int64              
 3   year              383160 non-null  float64            
 4   manufacturer      383160 non-null  str                
 5   model             383160 non-null  object             
 6   condition         372409 non-null  str                
 7   cylinders         372297 non-null  str                
 8   fuel              380990 non-null  str                
 9   odometer          380952 non-null  float64            
 10  title_status      375821 non-null  str                
 11  transmission      381491 non-null  str                
 12  drive             370746 non-null  str                
 13  

In [32]:
final_df.to_parquet("../data/raw/final_df.parquet")